# Quantum Phase Estimation — Amazon Braket

QPE estimates the phase $\phi$ in the eigenvalue equation
$U|\psi\rangle = e^{2\pi i \phi}|\psi\rangle$.
With $n$ precision qubits, it determines $\phi$ to $n$ bits of accuracy.

We use $U = Z$ (Pauli-Z) on $|1\rangle$, which has eigenvalue
$-1 = e^{i\pi}$, so $\phi = 1/2$.

In [ ]:
import cmath
import json

from braket.circuit import Circuit
from braket.devices import LocalSimulator

In [ ]:
device = LocalSimulator()

## QPE circuit builder

In [ ]:
def _add_inverse_qft(circuit, n):
    """Add inverse QFT to qubits 0..n-1."""
    for i in range(n - 1, -1, -1):
        for j in range(i + 1, n):
            k = j - i + 1
            angle = -2.0 * cmath.pi / (2**k)
            circuit.cphaseshift(j, i, angle)
        circuit.h(i)

def qpe_circuit(precision_bits=2):
    """Build QPE with controlled-Z^k rotations."""
    circuit = Circuit()
    n = precision_bits
    target = n

    circuit.x(target)
    for i in range(n):
        circuit.h(i)

    for k in range(n):
        power = 2**k
        circuit.cphaseshift(k, target, power * cmath.pi)

    _add_inverse_qft(circuit, n)

    for i in range(n):
        circuit.measure(i)
    return circuit

## QPE with 2 precision bits

Estimating $\phi = 1/2 = 0.1_2$ in binary.

In [ ]:
circuit = qpe_circuit(precision_bits=2)
print(circuit)

result = device.run(circuit, shots=1000).result()
counts = result.result_types[0].value

print(f"counts: {counts}")
for bits, n_shots in sorted(counts.items()):
    measured_phi = int(bits, 2) / 4
    print(f"  |{bits}> -> phi = {measured_phi:.2f}")
print("Expected: phi = 0.50")

## QPE with 3 precision bits

More precision qubits give higher resolution.

In [ ]:
circuit = qpe_circuit(precision_bits=3)

result = device.run(circuit, shots=1000).result()
counts = result.result_types[0].value

print(f"counts: {counts}")
for bits, n_shots in sorted(counts.items()):
    measured_phi = int(bits, 2) / 8
    print(f"  |{bits}> -> phi = {measured_phi:.3f}")
print("Expected: phi = 0.500")

## QPE on a different unitary

Using the phase gate $R(\pi/3)$ with eigenvalue $e^{i\pi/3}$,
so $\phi = 1/6 \approx 0.1667$.

In [ ]:
n = 3
target = n
theta = cmath.pi / 3

circuit = Circuit()
circuit.x(target)
for i in range(n):
    circuit.h(i)

for k in range(n):
    power = 2**k
    circuit.cphaseshift(k, target, power * theta)

_add_inverse_qft(circuit, n)
for i in range(n):
    circuit.measure(i)

result = device.run(circuit, shots=1000).result()
counts = result.result_types[0].value

print(f"counts: {counts}")
for bits, n_shots in sorted(counts.items()):
    measured_phi = int(bits, 2) / 8
    print(f"  |{bits}> -> phi = {measured_phi:.4f}")
print("Expected: phi = 0.1667")